# Prompt Engineering

Prompt Engineering basicaly means to iterate over a prompt, aiming to improve it and get better quality outputs.

For this we'll of course use an evaluation pipeline. Before I did not create a class for this, but now it should be really handy.

In [21]:
import json
from litellm import Message, completion
from constants import MODEL, DATASET_MODEL, GRADING_MODEL


class Evaluator:
    def __init__(
            self,
            prompt: str,
            model: str = MODEL,
            dataset_model: str = DATASET_MODEL,
            grading_model: str = GRADING_MODEL
        ):
        self.prompt = prompt
        self.model = model
        self.dataset_model = dataset_model
        self.grading_model = grading_model

    def generate_dataset(self, example: dict, output_file: str, count: int):
        prompt = f"""
Generate a test dataset for prompt evaluation.

The test cases in this dataset should be created  with this prompt in mind:
"{self.prompt}"

Example output:
```json
[
    {json.dumps(example)},
    ...
]
```

Output should be plain JSON.

Generate {count} objects.
        """
        message = Message(
            role="user",
            content=prompt
        )
        response = completion(
            model=self.dataset_model,
            messages=[message],
            format={
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {k: {"type": "string"} for k in example},
                    "required": list(example.keys()),
                }
            }
        )
        raw_json = response.choices[0].message.content
        json_response = json.loads(raw_json)
        with open(output_file, "w") as f:
            json.dump(json_response, f, indent=2)
        return json_response

    def run_prompt(self, test_case) -> dict:
        prompt = self.prompt + json.dumps(test_case)
        response = completion(
            model=self.model,
            messages=[Message(role="user", content=prompt)]
        )
        return {
            "test_case": test_case,
            "output": response.choices[0].message.content
        }

    def run_dataset(self, dataset):
        return [self.run_prompt(test_case) for test_case in dataset]

    def eval_output(self, result):
        test_case = result["test_case"]
        output = result["output"]
    
        eval_prompt = f"""
### Instruction
You are a professional personal trainer.
Your task is to evaluate the workout routine based on the user weight, height, age (optional) and goals.

<user_information_and_goals>
{json.dumps(test_case)}
</user_information_and_goals>

<model_output_to_be_graded>
{output}
</model_output_to_be_graded>

### Evaluation Criteria
1. Workout plan covers a full week of training
2. Workout plan includes rest days
3. Workout plan doesn't suggest exercises that might cause injuries based on the user weight
4. Workout plan includes exercises that best align with the user goals
5. Workout plan estimated time aligns with the users available time, if user specified it
6. Workout plan considers unavailable equipment

Only evaluate based on the criteria listed above.
Do not penalize for missing information that is not explicitly specified in criteria.

<score_anchors>
10: Criteria is met with no meaningful gaps
9-8: Criteria is met with minor gaps
7-6: Criteria partially met with considerable gaps
5-1: Criteria is barelly met with major gaps
</score_anchors>

Provide your evaluation as a structured JSON object with:
- "strengths": An array of strengths
- "weaknesses": An array of areas for improvement  
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10, based on the score anchors

Example Output:
{{
    "strengths": [...]
    "weaknesses": [...]
    "reasoning": <reasoning>
    "score": <score>
}}
"""
        response = completion(
            model=self.grading_model,
            messages=[Message(role="user", content=eval_prompt), Message(role="assistant", content="```json\n")],
            stop=["```"],
            format={
                "type": "object",
                "properties": {
                    "strengths": {"type": "array", "items": {"type": "string"}},
                    "weaknesses": {"type": "array", "items": {"type": "string"}},
                    "reasoning": {"type": "string"},
                    "score": {"type": "integer"},
                },
                "required": ["strengths", "weaknesses", "reasoning", "score"],
            },
            temperature=0
        )
        result["grade"] = json.loads(response.choices[0].message.content)
        return result

    def eval_dataset(self, dataset):
        final_results = []
        score_sum = 0
        results = self.run_dataset(dataset)
        for result in results:
            new_result = self.eval_output(result)
            final_results.append(new_result)
            score_sum += new_result["grade"]["score"]

        final_score = score_sum / len(results)

        return final_results, final_score

In [22]:
example = {
    "weight": "110.5kg",
    "height": "1.73m",
    "age": "23",
    "goal": "I don't want to loose much weight, but I want to improve the fat x muscle ratio.",
    "restrictions": "My gym doesn't have treadmills and I only have 1 hour available to workout."
}
prompt = "I was looking for some help to achieve my goal. Based on the following info, can you help me?\n"
evaluator = Evaluator(prompt=prompt)
dataset = evaluator.generate_dataset(example, "datasets/prompt_engineering.json", 3)
results, score = evaluator.eval_dataset(dataset)

print("Results:", results)
print("Score:", score)

Results: [{'test_case': {'weight': '80kg', 'height': '1.65m', 'age': '35', 'goal': "I want to improve my overall health and fitness, I'm open to any suggestions.", 'restrictions': 'I have a busy schedule and only have 30 minutes for workouts.'}, 'output': 'Absolutely! Based on your goals, time constraints, and current stats, here\'s a **comprehensive plan** to improve your overall health and fitness in 30 minutes a day. This approach balances **cardio, strength, and recovery** while being adaptable to your busy schedule.\n\n---\n\n### **1. Weekly Workout Plan (30 Minutes/Day)**\n**Focus:** Time-efficient, full-body workouts with minimal equipment (bodyweight or light weights if available).  \n**Structure:** 3-4 days of **HIIT/Strength** + 1-2 days of **Active Recovery** (walking, yoga, stretching).\n\n#### **Sample 30-Minute HIIT/Strength Session (Repeat 3-4x/Week):**\n- **Warm-Up (5 minutes):**  \n  - Jumping jacks (1 min)  \n  - Dynamic stretches (leg swings, arm circles, hip openers

## Being Clear and Direct

The idea here is simple.

### Clear Communication

- Using simple language that anyone can understand;
- State exactly what you want;
- Lead with straightforward statement about the task.

**Example:**
- **DON'T:** I was reading that most of LLM processing happens on the GPUs. How does that work?
- **DO:** Explain how GPUs are used for LLM training and inference.

### Direct Instructions

- Use instructions, not questions;
- Start with direct ACTION verbs ("write", "create", "generate"):

**Example:**
- **DON'T:** What should I do to learn more about how GPUs work?
- **DO:** Create a study plan around GPUs.

In [24]:
prompt = "Create a weekly workout routine, based on the following info about this person:\n"
evaluator = Evaluator(prompt=prompt)
with open("datasets/prompt_engineering.json", "r") as f:
    dataset = json.load(f)

results, score = evaluator.eval_dataset(dataset)

print("Results:", results)
print("Score:", score)

Results: [{'test_case': {'weight': '80kg', 'height': '1.65m', 'age': '35', 'goal': "I want to improve my overall health and fitness, I'm open to any suggestions.", 'restrictions': 'I have a busy schedule and only have 30 minutes for workouts.'}, 'output': 'Here’s a **30-minute weekly workout routine** tailored to your goals, schedule, and fitness level. This plan balances **strength, cardio, and flexibility** while keeping each session efficient and adaptable to a busy lifestyle.\n\n---\n\n### **Weekly Workout Plan (5 Days/Week)**  \n**Note:** Each session includes a warm-up, main workout, and cool-down. Adjust intensity based on your fitness level.\n\n---\n\n#### **Day 1: Full-Body Strength + Core (30 Minutes)**  \n**Warm-Up (5 min):**  \n- Jumping jacks (2 min)  \n- Dynamic stretches (leg swings, arm circles, torso twists) (3 min)  \n\n**Main Workout (20 min):**  \nPerform **3 rounds** of the following circuit (rest 30 seconds between rounds):  \n1. **Bodyweight Squats** – 15 reps  \

## Being specific

This basically means to add some guidelines for the model to follow.

They can be divided into two different types:

### Quality Guidelines

This means to adding qualities about the output.

**Example:**
- **Before:**
    ```text
    Write a story about a character who discovers a hidden talent.
    ```

- **After:**
    ```text
    Write a story about a character who discovers a hidden talent.

    Guidelines:
    1. Keep the story under 1000 words
    2. Include a clear action that reveals the character's talent
    3. Include at least one supporting character
    ```

### Procedure Guidelines

This one means to describe the steps that the model should follow.

**Example:**
- **Before:**
    ```text
    Write a story about a character who discovers a hidden talent.
    ```

- **After:**
    ```text
    Write a story about a character who discovers a hidden talent.

    Steps to follow:
    1. Brainstorm 3 talents that would create dramatic tension
    2. Pick the most insteresting talent
    3. Outline a pivotal scene that reveals the talent
    4. Brainstorm 3 supporting characters types that could increase the impact of this discovery
    ```

### When to use them

We should always try to use **guidelines**. This is highly valuable for the prompt's output.

For the **steps**, they are specially valuable when working with complex problems.

No need to say we can also use both approaches together.

In [25]:
prompt = """
Create a weekly workout routine, based on the user information and goals.

Guidelines:
1. The workout routine routine must cover a full week of training, including rest days
2. The workout should provide flexibility on exercises, taking into account what equipments might be available
3. The workout should provide flexibility on exercises, taking into account the time the user has available to workout
4. The workout must provide guidance on how to safely progress weights
5. The workout must take into account the users age, weight and height when suggesting exercises to avoid injuries
6. The workout must take into account the users age, weight and height when suggesting exercises for ease of execution

User Information:

"""
evaluator = Evaluator(prompt=prompt)
with open("datasets/prompt_engineering.json", "r") as f:
    dataset = json.load(f)

results, score = evaluator.eval_dataset(dataset)

print("Results:", results)
print("Score:", score)

Results: [{'test_case': {'weight': '80kg', 'height': '1.65m', 'age': '35', 'goal': "I want to improve my overall health and fitness, I'm open to any suggestions.", 'restrictions': 'I have a busy schedule and only have 30 minutes for workouts.'}, 'output': '### **Weekly Workout Routine for Overall Health & Fitness (30 Minutes/Day)**  \n**User Profile:** 35 years old, 80 kg, 1.65 m, busy schedule, no equipment preference, injury prevention focus.  \n\n---\n\n### **Weekly Structure**  \n**Rest Days:** Saturday, Sunday (active recovery recommended).  \n**Workout Days:** Monday–Friday (30 minutes/day).  \n**Focus:** Full-body strength, cardiovascular health, flexibility, and injury prevention.  \n\n---\n\n### **Monday: Full-Body Strength (Bodyweight & Light Resistance)**  \n**Warm-up (5 min):**  \n- Jumping jacks (2 min)  \n- Dynamic stretches (leg swings, arm circles, torso twists)  \n\n**Workout (20 min):**  \n1. **Bodyweight Squats** (3 sets of 15 reps) – *Targets legs, glutes, core.*  \

## Structure with XML Tags

The idea here is simple, create content boundaries that the model can easily identify. We do that throught custom XML tags.

**Example:**
- **Before:**
    ```text
    Create a summary about my software documentation.

    {docs}

    Guidelines:
    1. Keep the summary under 1000 words
    2. ...
    ```

- **After:**
    ```text
    Create a summary about my software documentation.

    <software_documentation>
    {docs}
    </software_documentation>

    Guidelines:
    1. Keep the summary under 1000 words
    2. ...
    ```


This templating is specially relevent when:
- Using large amounts of context or data
- Mixing code, documentation, data...
- Being extra clear about content boundaries
- Working with complex prompts that interpolate multiple variables 

In [28]:
import json
from litellm import Message, completion
from constants import MODEL, DATASET_MODEL, GRADING_MODEL


class Evaluator:
    def __init__(
            self,
            prompt: str,
            model: str = MODEL,
            dataset_model: str = DATASET_MODEL,
            grading_model: str = GRADING_MODEL
        ):
        self.prompt = prompt
        self.model = model
        self.dataset_model = dataset_model
        self.grading_model = grading_model

    def generate_dataset(self, example: dict, output_file: str, count: int):
        prompt = f"""
Generate a test dataset for prompt evaluation.

The test cases in this dataset should be created  with this prompt in mind:
"{self.prompt}"

Example output:
```json
[
    {json.dumps(example)},
    ...
]
```

Output should be plain JSON.

Generate {count} objects.
        """
        message = Message(
            role="user",
            content=prompt
        )
        response = completion(
            model=self.dataset_model,
            messages=[message],
            format={
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {k: {"type": "string"} for k in example},
                    "required": list(example.keys()),
                }
            }
        )
        raw_json = response.choices[0].message.content
        json_response = json.loads(raw_json)
        with open(output_file, "w") as f:
            json.dump(json_response, f, indent=2)
        return json_response

    def run_prompt(self, test_case) -> dict:
        prompt = self.prompt.format(user_info=json.dumps(test_case))
        response = completion(
            model=self.model,
            messages=[Message(role="user", content=prompt)]
        )
        return {
            "test_case": test_case,
            "output": response.choices[0].message.content
        }

    def run_dataset(self, dataset):
        return [self.run_prompt(test_case) for test_case in dataset]

    def eval_output(self, result):
        test_case = result["test_case"]
        output = result["output"]
    
        eval_prompt = f"""
### Instruction
You are a professional personal trainer.
Your task is to evaluate the workout routine based on the user weight, height, age (optional) and goals.

<user_information_and_goals>
{json.dumps(test_case)}
</user_information_and_goals>

<model_output_to_be_graded>
{output}
</model_output_to_be_graded>

### Evaluation Criteria
1. Workout plan covers a full week of training
2. Workout plan includes rest days
3. Workout plan doesn't suggest exercises that might cause injuries based on the user weight
4. Workout plan includes exercises that best align with the user goals
5. Workout plan estimated time aligns with the users available time, if user specified it
6. Workout plan considers unavailable equipment

Only evaluate based on the criteria listed above.
Do not penalize for missing information that is not explicitly specified in criteria.

<score_anchors>
10: Criteria is met with no meaningful gaps
9-8: Criteria is met with minor gaps
7-6: Criteria partially met with considerable gaps
5-1: Criteria is barelly met with major gaps
</score_anchors>

Provide your evaluation as a structured JSON object with:
- "strengths": An array of strengths
- "weaknesses": An array of areas for improvement  
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10, based on the score anchors

Example Output:
{{
    "strengths": [...]
    "weaknesses": [...]
    "reasoning": <reasoning>
    "score": <score>
}}
"""
        response = completion(
            model=self.grading_model,
            messages=[Message(role="user", content=eval_prompt), Message(role="assistant", content="```json\n")],
            stop=["```"],
            format={
                "type": "object",
                "properties": {
                    "strengths": {"type": "array", "items": {"type": "string"}},
                    "weaknesses": {"type": "array", "items": {"type": "string"}},
                    "reasoning": {"type": "string"},
                    "score": {"type": "integer"},
                },
                "required": ["strengths", "weaknesses", "reasoning", "score"],
            },
            temperature=0
        )
        result["grade"] = json.loads(response.choices[0].message.content)
        return result

    def eval_dataset(self, dataset):
        final_results = []
        score_sum = 0
        results = self.run_dataset(dataset)
        for result in results:
            new_result = self.eval_output(result)
            final_results.append(new_result)
            score_sum += new_result["grade"]["score"]

        final_score = score_sum / len(results)

        return final_results, final_score

In [ ]:
prompt = """
Create a weekly workout routine, based on the user information and goals.

<user_information_and_goals>
{user_info}
</user_information_and_goals>

### Guidelines:
1. The workout routine must cover 1 week of training, including rest days
2. The workout must provide alternative exercises alternatives to avoid unavailable equipments
3. The workout should provide flexibility on exercises, taking into account the time the user has available to workout
4. The workout must provide guidance on how to safely progress weights
5. The workout must take into account the users age, weight and height when suggesting exercises to avoid injuries
6. The workout must take into account the users age, weight and height when suggesting exercises for ease of execution
"""
evaluator = Evaluator(prompt=prompt)
with open("datasets/prompt_engineering.json", "r") as f:
    dataset = json.load(f)

results, score = evaluator.eval_dataset(dataset)

print("Results:", results)
print("Score:", score)

Results: [{'test_case': {'weight': '80kg', 'height': '1.65m', 'age': '35', 'goal': "I want to improve my overall health and fitness, I'm open to any suggestions.", 'restrictions': 'I have a busy schedule and only have 30 minutes for workouts.'}, 'output': '### **Weekly Workout Routine for Overall Health & Fitness**  \n**User Profile:** 35 years old, 80 kg, 1.65 m, 30-minute workouts, no equipment preferred.  \n**Goal:** Improve overall health and fitness with minimal time and equipment.  \n**Focus:** Full-body strength, cardiovascular health, flexibility, and injury prevention.  \n\n---\n\n### **Weekly Structure**  \n**Rest Days:** Wednesday & Sunday  \n**Active Recovery:** Wednesday (light stretching/yoga)  \n**Workout Days:** Monday, Tuesday, Thursday, Friday, Saturday  \n\n---\n\n### **Monday: Upper Body Strength + Core (30 min)**  \n**Warm-up (5 min):** Jumping jacks (2 min), arm circles (1 min), cat-cow stretch (2 min).  \n**Circuit (Repeat 3x with 30 sec rest between rounds):**  

## Providing Examples

To further improve the ouput, we can provide a concrete pair of input/output.

This is also know as:
- **One-Shot:** Provide a single example
- **Multi-Shot:** Provide multiple examples

Also, we should add extra context about why the output is good.

### Best Practices
- Be clear on what you're showing to the model, don't just drop the input and output example
- Explain **WHY** the output is considered ideal
- Use XML templating
- **Add examples to address edge cases**

In [36]:
with open("datasets/ideal_output_example.txt") as f:
    ideal_output = f.read()

with open("datasets/prompt_engineering.json", "r") as f:
    dataset = json.load(f)

prompt = f"""
Create a weekly workout routine, based on the user information and goals.

<user_information_and_goals>
{{user_info}}
</user_information_and_goals>

### Guidelines:
1. The workout routine must cover 1 week of training, including rest days
2. The workout must provide alternative exercises alternatives to avoid unavailable equipments
3. The workout should provide flexibility on exercises, taking into account the time the user has available to workout
4. The workout must provide guidance on how to safely progress weights
5. The workout must take into account the users age, weight and height when suggesting exercises to avoid injuries
6. The workout must take into account the users age, weight and height when suggesting exercises for ease of execution

Here's an example of an ideal response:

<sample_input>
{{{{
    "weight": "90kg",
    "height": "1.75m",
    "age": "28",
    "goal": "I want to build muscle, but I'm not sure how to do it effectively.",
    "restrictions": "I don't have much time to work out and my gym only offers bodyweight exercises."
}}}}
</sample_input>

<ideal_output>
{ideal_output}
</ideal_output>

This output is good because:
- The workout plan covers a full week of training with specific days for different muscle groups
- Rest days are included (Saturday and Sunday) or active recovery
- The plan includes exercises that align with the user's goal of building muscle using bodyweight exercises
- The plan is time-efficient, with sessions estimated to be 45-60 minutes, which fits the user's limited time
- The plan considers the user's equipment restrictions by focusing on bodyweight exercises
"""
evaluator = Evaluator(prompt=prompt)
results, score = evaluator.eval_dataset(dataset)

print("Results:", results)
print("Score:", score)

Results: [{'test_case': {'weight': '80kg', 'height': '1.65m', 'age': '35', 'goal': "I want to improve my overall health and fitness, I'm open to any suggestions.", 'restrictions': 'I have a busy schedule and only have 30 minutes for workouts.'}, 'output': 'Here’s a **time-efficient, full-body weekly workout routine** tailored to your goal of **improving overall health and fitness** with a **30-minute time limit** and **no equipment**. This plan prioritizes **joint-friendly movements**, **muscle balance**, and **cardiovascular health**, while accounting for your age, weight, and height to minimize injury risk and maximize effectiveness.\n\n---\n\n### **Weekly Workout Split**  \n**Days:** Monday, Tuesday, Wednesday, Thursday, Friday (rest on Saturday/Sunday or active recovery)  \n**Focus:** Full-body strength, mobility, and active recovery.\n\n---\n\n### **Day 1: Full-Body Strength + Core (Upper + Lower)**  \n**Warm-up (5 min):** Arm circles, leg swings, cat-cow stretches, light marching